In [1]:
import mlflow
import pandas as pd
import numpy as np
import re
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import shap
import matplotlib.pyplot as plt

/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Steps to improve model:

1. Compare training and testing scores

2. Use feature importantance to identity relevant features, and use correlation to identity any colinearity

In [ ]:
# db_path = os.path.abspath("mlflow.db")
# mlflow.set_tracking_uri(f"sqlite:///{db_path}")
# print(f"Tracking URI: {mlflow.get_tracking_uri()}")
# mlflow.set_experiment("vegas_odds_testing")

Tracking URI: sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db


<Experiment: artifact_location=('/Users/abhaybapat/Desktop/Data Science '
 'Project/betting_classification_model/notebooks/mlruns/14'), creation_time=1780425202566, experiment_id='14', last_update_time=1780425202566, lifecycle_stage='active', name='dataset_testing', tags={}, trace_location=None, workspace='default'>

In [43]:
df = pd.read_csv("../data/final_dataset.csv", index_col=0)

In [47]:
df["teamName_away"].unique()

array(['Suns', 'Pistons', 'Kings', 'Nuggets', 'Trail Blazers', 'Celtics',
       'Mavericks', 'Bulls', 'Hawks', 'Knicks', 'Rockets', 'Clippers',
       'Nets', 'Spurs', 'Bucks', 'Lakers', '76ers', 'Pacers', 'Cavaliers',
       'Wizards', 'Jazz', 'Warriors', 'Thunder', 'Hornets', 'Heat',
       'Timberwolves', 'Magic', 'Raptors', 'Grizzlies', 'Pelicans'],
      dtype=object)

In [38]:
vegas_odds = pd.read_csv("../data/2023-2026_Odds - Sheet1.csv")

In [39]:
vegas_odds["date"] = pd.to_datetime(vegas_odds["date"])
vegas_odds = vegas_odds.sort_values(by="date")

In [40]:
vegas_odds["win_home"] = np.where(vegas_odds["score_home"] > vegas_odds["score_away"], 1, 0)

In [41]:
vegas_odds

,season,date,home,away,moneyline_home,moneyline_away,score_home,score_away,win_home
1317,2025,2024-10-22,Boston Celtics,New York Knicks,-263,210,132,109,1
1316,2025,2024-10-22,Los Angeles Lakers,Minnesota Timberwolves,100,-120,110,103,1
1314,2025,2024-10-23,Philadelphia 76ers,Milwaukee Bucks,150,-179,109,124,0
1315,2025,2024-10-23,Detroit Pistons,Indiana Pacers,179,-238,109,115,0
1307,2025,2024-10-23,Los Angeles Clippers,Phoenix Suns,165,-200,113,116,0
...,...,...,...,...,...,...,...,...,...
4,2025,2025-06-11,Indiana Pacers,Oklahoma City Thunder,185,-222,116,107,1
3,2025,2025-06-13,Indiana Pacers,Oklahoma City Thunder,195,-238,104,111,0
2,2025,2025-06-16,Oklahoma City Thunder,Indiana Pacers,-385,309,120,109,1
1,2025,2025-06-19,Indiana Pacers,Oklahoma City Thunder,185,-222,108,91,1


In [ ]:
df["game_date"] = pd.DataFrame(df["game_date"])

df = df.sort_values(by="game_date")

In [31]:
df = df.iloc[:49942]

In [32]:
df

,game_date,pre_game_elo_home,is_B2B_home,pre_game_elo_away,is_B2B_away,pre_game_elo_diff,days_rest_diff,possessions_rolling_diff,eFG_rolling_diff,TO%_rolling_diff,OREB%_rolling_diff,FTR_rolling_diff,off_rating_rolling_diff,def_rating_rolling_diff,net_rating_rolling_diff,win_home
0,1986-11-01 20:00:00,1421.87,True,1448.97,True,-27.10,0.0,-10.982400,-0.099275,-0.086457,-0.091463,-0.002415,-8.464992,-3.910563,-4.554429,True
1,1986-11-01 20:00:00,1431.30,True,1518.94,True,-87.64,0.0,-8.064000,0.094880,-0.027357,-0.040309,-0.089677,8.659146,-2.711930,11.371076,True
2,1986-11-01 20:00:00,1471.80,True,1463.65,True,8.15,0.0,0.998400,0.161866,0.007656,-0.051724,-0.095930,18.519900,2.888894,15.631006,True
3,1986-11-01 20:00:00,1443.05,True,1524.34,True,-81.29,0.0,4.992000,-0.044118,0.040420,-0.180180,0.058229,-16.170137,16.614965,-32.785102,True
4,1986-11-01 20:00:00,1496.85,True,1461.72,True,35.13,0.0,4.723200,-0.195382,0.000538,-0.018490,-0.147186,-36.429595,-28.609532,-7.820063,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49937,2025-06-11 20:30:00,1667.78,False,1770.29,False,-102.51,0.0,0.092651,0.022576,0.036701,-0.045782,0.023428,-3.931002,5.415300,-9.346302,True
49938,2025-06-13 20:30:00,1678.46,False,1759.61,False,-81.15,0.0,0.159587,0.025552,0.022350,-0.048702,-0.002818,-1.619665,2.633134,-4.252799,False
49939,2025-06-16 20:30:00,1767.94,False,1670.13,False,97.81,0.0,-1.317481,-0.020935,-0.020275,0.063770,0.015884,4.075105,-2.082664,6.157769,True
49940,2025-06-19 20:30:00,1665.18,False,1772.89,False,-107.71,0.0,0.924339,0.020455,0.037775,-0.045532,-0.008372,-5.232860,3.987117,-9.219977,True


In [ ]:
X = df.drop(columns=["game_date", "win_home"])  
y = df["win_home"]

In [42]:
def convert_american_to_implied(odds):
    """
    Converts American odds (e.g., -150, +130) to implied probabilities.
    """
    return np.where(odds < 0, 
                    abs(odds) / (abs(odds) + 100), 
                    100 / (odds + 100))

# IMPORTANT: Change 'home_odds' and 'away_odds' to match the actual column names in your dataframe
vegas_odds['home_implied'] = convert_american_to_implied(vegas_odds['moneyline_home'])
vegas_odds['away_implied'] = convert_american_to_implied(vegas_odds['moneyline_away'])

TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
# baseline_pipeline = Pipeline([
#     ("scaler", StandardScaler()), 
#     ("lr", LogisticRegression(max_iter=1000, penalty='l2')) # L2 protects against VIF!
# ])

# tscv = TimeSeriesSplit(n_splits=5)
# param_grid = {"lr__C": [0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0]}

# # Change your run name dynamically or via a variable
# with mlflow.start_run(run_name="20_rolling"):
#     acc = []
#     fold_log_loss = []
#     fold_brier = []
    
#     for fold, (train_index, test_index) in enumerate(tscv.split(X)):
#         X_train = X.iloc[train_index]
#         y_train = y.iloc[train_index]
#         X_test = X.iloc[test_index]
#         y_test = y.iloc[test_index]
        
#         # 1. Use TimeSeriesSplit to find the best hyperparameter on ALL available train data
#         inner_cv = TimeSeriesSplit(n_splits=3)
#         grid = GridSearchCV(baseline_pipeline, param_grid, scoring='neg_log_loss', cv=inner_cv)
#         grid.fit(X_train, y_train)
        
#         # 2. Calibrate using 'prefit' by splitting temporally just for calibration, 
#         # OR use CalibratedClassifierCV's built-in cv with a TimeSeriesSplit.
#         # For sports, a clean approach is calibrating on the training data directly using out-of-fold hooks,
#         # but since it's linear, pass cv='prefit' after a small temporal split:
        
#         cv_split = int(len(X_train) * 0.85)
#         X_train_internal = X_train.iloc[:cv_split]
#         y_train_internal = y_train.iloc[:cv_split]
#         X_val_internal = X_train.iloc[cv_split:]
#         y_val_internal = y_train.iloc[cv_split:]
        
#         # Refit best model on internal train
#         best_lr = grid.best_estimator_.fit(X_train_internal, y_train_internal)
        
#         # Calibrate on the validation chunk
#         frozen_model = FrozenEstimator(best_lr)
#         calibrated_model = CalibratedClassifierCV(estimator=frozen_model, method='sigmoid')        
#         calibrated_model.fit(X_val_internal, y_val_internal)
        
#         # 3. Predict on unseen future test fold
#         y_pred = calibrated_model.predict(X_test)
#         y_prob = calibrated_model.predict_proba(X_test)[:, 1]
        
#         # Evaluate
#         acc.append(accuracy_score(y_test, y_pred))
#         fold_log_loss.append(log_loss(y_test, y_prob))
#         fold_brier.append(brier_score_loss(y_test, y_prob))
        
#     # FIX: Ensure we take the mean of the entire list, not just the last fold!
#     mlflow.log_metric("mean_acc", np.mean(acc))
#     mlflow.log_metric("mean_log_loss", np.mean(fold_log_loss))
#     mlflow.log_metric("mean_brier_score_loss", np.mean(fold_brier))

/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/abhaybapat/Desktop/Data Science Project/betting_classification_

In [80]:
# models = {
#     "RandomForest": Pipeline([("clf", RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=20))]),
#     "XGBoost": Pipeline([("clf", XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, reg_lambda=10))]),
#     "LGBM": Pipeline([("clf", LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, reg_lambda=10))]),
#     "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(C=0.1))])
# }

In [81]:
# tscv = TimeSeriesSplit(n_splits=5)
# for model_name, pipeline in models.items():
#     with mlflow.start_run(run_name=model_name):
#         fold_log_loss = []
#         fold_brier = []
        
#         for fold, (train_index, test_index) in enumerate(tscv.split(X)):
#             X_train_full = X.iloc[train_index]
#             y_train_full = y.iloc[train_index]
#             X_test = X.iloc[test_index]
#             y_test = y.iloc[test_index]
        
#             calib_size = int(len(X_train_full) * 0.2)
#             X_fit = X_train_full[:-calib_size]
#             y_fit = y_train_full[:-calib_size]
#             X_calib = X_train_full[-calib_size:]
#             y_calib = y_train_full[-calib_size:]
            
#             pipeline.fit(X_fit, y_fit)
#             frozen_model = FrozenEstimator(pipeline)

#             calibrated_model = CalibratedClassifierCV(estimator=frozen_model, method='sigmoid')
#             calibrated_model.fit(X_calib, y_calib)
            
#             y_prob = calibrated_model.predict_proba(X_test)[:, 1]
            
#             fold_log_loss.append(log_loss(y_test, y_prob))
#             fold_brier.append(brier_score_loss(y_test, y_prob))
            
#             mlflow.log_metric(f"fold_{fold}_log_loss", fold_log_loss[-1])
#             mlflow.log_metric(f"fold_{fold}_brier", fold_brier[-1])
        
#         mlflow.log_metric("mean_log_loss", np.mean(fold_log_loss))
#         mlflow.log_metric("mean_brier_score_loss", np.mean(fold_brier))  
              
#         mlflow.sklearn.log_model(calibrated_model, name="calibrated_model")

mlflow ui --backend-store-uri "sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db" --port 5001